# Training TopoBench on A123 Auditory Cortex Data with CWN

This notebook demonstrates graph-level classification on the A123 mouse auditory cortex dataset using a CWN backbone from TopoModelX inside the TopoBench `TBModel` workflow.

The goal is to predict the best-frequency bin (`bf_bin`) from graph structure built from neural correlation matrices.

## Contribution context

The A123 processing code builds one graph per `(session, frequency-bin)` by pooling neurons across cortical layers, deduplicating neuron indices, and slicing the shared signal/noise correlation matrices. This tutorial shows how those graphs can be lifted to cell complexes and consumed by a CWN model.

In [ ]:
import torch
import lightning as pl
from functools import partial
from omegaconf import OmegaConf

from topobench.data.loaders.graph.a123_loader import A123DatasetLoader
from topobench.dataloader.dataloader import TBDataloader
from topobench.data.preprocessor import PreProcessor
from topobench.model.model import TBModel
from topobench.loss.loss import TBLoss
from topobench.optimizer import TBOptimizer
from topobench.evaluator.evaluator import TBEvaluator
from topobench.nn.encoders import AllCellFeatureEncoder
from topobench.nn.readouts import PropagateSignalDown
from topobench.nn.wrappers.cell.cwn_wrapper import CWNWrapper
from topomodelx.nn.cell.cwn import CWN

In [ ]:
loader_config = OmegaConf.create({
    'data_domain': 'graph',
    'data_type': 'A123',
    'data_name': 'a123_cortex_m',
    'data_dir': './data/a123/',
    'corr_threshold': 0.3,
    'specific_task': 'classification',
})

transform_config = OmegaConf.create({
    'transform_type': 'lifting',
    'transform_name': 'CellCycleLifting',
    'complex_dim': 3,
    'max_cell_length': 18,
    'feature_lifting': 'ProjectionSum',
    'preserve_edge_attr': False,
})

split_config = OmegaConf.create({
    'learning_setting': 'inductive',
    'split_type': 'random',
    'data_seed': 0,
    'data_split_dir': './data/a123/splits/',
    'train_prop': 0.5,
})

In [ ]:
graph_loader = A123DatasetLoader(loader_config)
dataset, dataset_dir = graph_loader.load()

preprocessor = PreProcessor(dataset, dataset_dir, transform_config)
dataset_train, dataset_val, dataset_test = preprocessor.load_dataset_splits(split_config)

datamodule = TBDataloader(dataset_train, dataset_val, dataset_test, batch_size=32)

In [ ]:
sample = dataset_train[0]
print(type(sample))
if hasattr(sample, 'keys'):
    print(sample.keys())

In [ ]:
dim_hidden = 16
in_channels = 3
out_channels = 9
num_cell_dimensions = 3

feature_encoder = AllCellFeatureEncoder(
    in_channels=[in_channels] * num_cell_dimensions,
    out_channels=dim_hidden,
)

backbone = CWN(
    in_channels_0=dim_hidden,
    in_channels_1=dim_hidden,
    in_channels_2=dim_hidden,
    hid_channels=dim_hidden,
    n_layers=4,
)

backbone_wrapper = partial(
    CWNWrapper,
    out_channels=dim_hidden,
    num_cell_dimensions=num_cell_dimensions,
)

readout = PropagateSignalDown(
    readout_name='PropagateSignalDown',
    num_cell_dimensions=num_cell_dimensions,
    hidden_dim=dim_hidden,
    out_channels=out_channels,
    task_level='graph',
    pooling_type='sum',
)

loss = TBLoss(dataset_loss={'task': 'classification', 'loss_type': 'cross_entropy'})
optimizer = TBOptimizer(optimizer_id='Adam', parameters={'lr': 1e-3, 'weight_decay': 5e-4})
evaluator = TBEvaluator(task='classification', num_classes=out_channels, metrics=['accuracy', 'f1', 'precision', 'recall'])

In [ ]:
model = TBModel(
    backbone=backbone,
    backbone_wrapper=backbone_wrapper,
    readout=readout,
    loss=loss,
    feature_encoder=feature_encoder,
    optimizer=optimizer,
    evaluator=evaluator,
)

trainer = pl.Trainer(max_epochs=5, accelerator='auto', log_every_n_steps=1)
trainer.fit(model, datamodule=datamodule)
trainer.test(model, datamodule=datamodule)

## Results section

Record validated metrics here after running the notebook in a clean environment. Do not claim model performance until the final run logs are available.

Suggested table columns: split, accuracy, macro-F1, precision, recall, notes.